## Try to analyze sensitivity

In [ ]:
# constraints = cvxpy_med_problem.constraints
# print(constraints)
# for i in range(len(constraints)):
#     print(
#         f"\nThe optimal dual variable for constraint {i} is\n{np.round(constraints[i].dual_value, 4)}"
#     )

In [1]:
# Copy from Grok
import cvxpy as cp
import numpy as np
from scipy.linalg import svd

# Set random seed for reproducibility
np.random.seed(42)

# Problem data
n = 3  # Matrix dimension
C = np.random.randn(n, n)
C = (C + C.T) / 2  # Symmetric cost matrix
A1 = np.random.randn(n, n)
A1 = (A1 + A1.T) / 2  # Matrix for constraint 1
A2 = np.random.randn(n, n)
A2 = (A2 + A2.T) / 2  # Matrix for constraint 2
A3 = np.random.randn(n, n)
A3 = (A3 + A3.T) / 2  # Matrix for constraint 3

# Define the SDP variable
X = cp.Variable((n, n), symmetric=True)

# Objective: Minimize trace(C @ X)
objective = cp.Minimize(cp.trace(C @ X))

# Constraints
constraints = [
    cp.trace(A1 @ X) <= 1,  # Scalar constraint
    X - np.eye(n) <= 0,     # LMI constraint 1: X <= I
    X + np.eye(n) >= 0,     # LMI constraint 2: X >= -I
    A3 @ X + X @ A3.T <= 0, # LMI constraint 3: A3*X + X*A3^T <= 0
    X >> 0                  # X positive semidefinite
]

# Form and solve the problem
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.SCS, verbose=True)

# Check solver status
print("Solver status:", problem.status)
if problem.status not in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
    print("Problem is infeasible or unbounded. Check constraints.")
    exit()

# Extract dual values
dual_scalar = constraints[0].dual_value  # Scalar dual for trace(A1 @ X) <= 1
dual_LMI1 = constraints[1].dual_value    # Matrix dual for X <= I
dual_LMI2 = constraints[2].dual_value    # Matrix dual for X >= -I
dual_LMI3 = constraints[3].dual_value    # Matrix dual for A3*X + X*A3^T <= 0

# Ensure duals are symmetric (SCS may introduce small numerical asymmetries)
dual_LMI1 = (dual_LMI1 + dual_LMI1.T) / 2
dual_LMI2 = (dual_LMI2 + dual_LMI2.T) / 2
dual_LMI3 = (dual_LMI3 + dual_LMI3.T) / 2

# Analyze dual matrices using SVD
def analyze_dual_matrix(Lambda, name):
    U, S, Vt = svd(Lambda, full_matrices=True)
    frobenius_norm = np.linalg.norm(Lambda, 'fro')
    spectral_norm = np.max(S)
    rank = np.sum(S > 1e-6)  # Tolerance for numerical rank
    print(f"\n--- Analysis of Dual Matrix for {name} ---")
    print(f"Singular Values: {S}")
    print(f"Frobenius Norm: {frobenius_norm:.4f}")
    print(f"Spectral Norm: {spectral_norm:.4f}")
    print(f"Rank: {rank}")
    return frobenius_norm, rank

# Analyze each dual matrix
norm1, rank1 = analyze_dual_matrix(dual_LMI1, "Constraint 1 (X <= I)")
norm2, rank2 = analyze_dual_matrix(dual_LMI2, "Constraint 2 (X >= -I)")
norm3, rank3 = analyze_dual_matrix(dual_LMI3, "Constraint 3 (A3*X + X*A3^T <= 0)")

# Scalar dual for comparison
print("\nScalar Dual for trace(A1 @ X) <= 1:", dual_scalar)

# Compare constraint weights
norms = {"Constraint 1": norm1, "Constraint 2": norm2, "Constraint 3": norm3}
ranks = {"Constraint 1": rank1, "Constraint 2": rank2, "Constraint 3": rank3}
most_restrictive = max(norms, key=norms.get)
densest = max(ranks, key=ranks.get)
print(f"\nMost Restrictive Constraint (largest Frobenius norm): {most_restrictive} (norm = {norms[most_restrictive]:.4f})")
print(f"Densest Constraint (highest rank): {densest} (rank = {ranks[densest]})")

# Trade-off analysis (example: relax Constraint 1)
slack = cp.Variable()
objective_slack = cp.Minimize(cp.trace(C @ X) + 100 * slack)  # Penalize slack
constraints_slack = [
    cp.trace(A1 @ X) <= 1,
    X - np.eye(n) <= slack * np.eye(n),  # Relax X <= I to X <= I + slack*I
    X + np.eye(n) >= 0,
    A3 @ X + X @ A3.T <= 0,
    X >> 0,
    slack >= 0
]
problem_slack = cp.Problem(objective_slack, constraints_slack)
problem_slack.solve(solver=cp.SCS, verbose=True)
print("\nTrade-off Analysis (relaxing Constraint 1):")
print(f"Slack value: {slack.value:.4f}")
print(f"New objective value: {problem_slack.value:.4f}")
print(f"Original objective value: {problem.value:.4f}")

                                     CVXPY                                     
                                     v1.6.0                                    
(CVXPY) Aug 09 11:35:32 AM: Your problem has 9 variables, 37 constraints, and 0 parameters.
(CVXPY) Aug 09 11:35:32 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 09 11:35:32 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 09 11:35:32 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 09 11:35:32 AM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Aug 09 11:35:32 AM: Compiling problem (target solver=SCS).
(CVXP

## List of guesses
- Track constraints
- Add disturbance

## First guess

## Second guess


In [ ]:
# Our case